# ADARP stress-event analysis
Sohail Guller · BASS Emerging Scholars

Recovered August inventory, event-tag and figure-export sections plus a reconstructed modeling section. The five-minute random forest was checked against the original processed table. Matching and feature-table construction were not recovered. The continuous-alert and next-day analyses are separate from the primary experiment.

In [ ]:
from pathlib import Path
import os, json, platform
import numpy as np
import pandas as pd
import sklearn
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, average_precision_score
try:
    from IPython.display import display
except ImportError:
    display = print

# Set FEATURE_CSV to an explicit local path if your file is elsewhere.
FEATURE_CSV = os.environ.get("FEATURE_CSV")
import sys
MOUNT_DRIVE = "google.colab" in sys.modules  # In Colab, set True to read My Drive/ADARP_Project.
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
roots = [Path("data"), Path("/content/drive/MyDrive/ADARP_Project")]
hits = [p for root in roots if root.is_dir() for p in root.rglob("clean_modeling_dataset.csv")]
if not FEATURE_CSV and len(hits) > 1:
    raise ValueError("Multiple feature tables: set FEATURE_CSV to the intended path.")
path = Path(FEATURE_CSV) if FEATURE_CSV else next(iter(hits), None)
print("Mode:", "refit from feature table" if path else "reported results only")
print("Environment:", platform.python_version(), pd.__version__, np.__version__, sklearn.__version__)

## Recovered inventory and event tags
Source: Untitled0.ipynb, August 4–5. Original parsing logic retained; local output paths changed and participant previews omitted.

In [ ]:
Path('runs').mkdir(exist_ok=True)
project = Path('/content/drive/MyDrive/ADARP_Project')
zip_path = project / '01_Raw_Data/Sensor Data.zip'
if zip_path.is_file():
    import zipfile
    import pandas as pd

    with zipfile.ZipFile(zip_path, "r") as archive:
        inventory = pd.DataFrame([
            {
                "path": item.filename,
                "size_bytes": item.file_size,
                "compressed_bytes": item.compress_size
            }
            for item in archive.infolist()
            if not item.is_dir()
        ])

    print("Number of files:", len(inventory))


    inventory.to_csv("runs/sensor_inventory.csv", index=False)
    import zipfile
    import pandas as pd
    import re

    records = []

    with zipfile.ZipFile(zip_path, "r") as archive:
        for path in archive.namelist():
            if not path.endswith("/tags.csv"):
                continue

            participant_match = re.search(r"Part ([^/]+)", path)
            participant = participant_match.group(1)
            session = path.split("/")[-2]

            content = archive.read(path).decode("utf-8-sig").strip()

            if content:
                for line in content.splitlines():
                    timestamp = float(line.strip().split(",")[0])

                    records.append({
                        "participant": participant,
                        "session": session,
                        "unix_timestamp": timestamp,
                        "event_time_utc": pd.to_datetime(
                            timestamp, unit="s", utc=True
                        )
                    })

    tags = pd.DataFrame(records)

    print("Total event tags:", len(tags))



    tags.to_csv("runs/all_event_tags.csv", index=False)
else:
    print('Raw archive unavailable; continuing from the processed table.')

## Reported results
A positive window precedes a participant-marked stress event. Its control is an unmarked comparison period, not proof of no stress. The shortest window had the highest mean discrimination across held-out participants. This analysis does not test relapse prediction.

In [ ]:
windows = [5, 15, 30, 60]
reported = pd.Series([0.773870, 0.628, 0.634, 0.562], index=windows, name="reported_mean_auc")
display(reported.to_frame().rename_axis("window_minutes"))
print("Five-minute random forest: 11 participants; 207 marked and 207 matched controls.")
print("Participant-mean AUC: 0.774 (95% interval 0.694–0.851); pooled AUC: 0.759.")

## Input and validation
Start from the study's processed table. The 24 predictors summarize change from the immediately preceding equal-length baseline in four signals. Do not use heart-rate variability, participant IDs, outcome labels, or absolute timestamps as predictors. Raw signal alignment, matching and exclusion cells were not recovered; this notebook does not recreate them.

In [ ]:
features = [f"{signal}_{stat}_change" for signal in ("hr", "eda", "temp", "acc")
            for stat in ("mean", "std", "median", "min", "max", "slope")]
if path is not None:
    data = pd.read_csv(path)
    needed = set(features + ["window_minutes", "participant", "label"])
    if needed - set(data):
        raise ValueError(f"Missing original columns: {sorted(needed - set(data))}")
    if data[["participant", "window_minutes", "label"]].isna().any().any():
        raise ValueError("Missing grouping, window, or outcome value.")
    if set(data.window_minutes) != set(windows):
        raise ValueError("Unexpected monitoring windows.")
    data[features] = data[features].apply(pd.to_numeric, errors="raise")
    if np.isinf(data[features].to_numpy()).any():
        raise ValueError("Infinite feature value; inspect upstream preprocessing.")
    for w, d in data.groupby("window_minutes"):
        if d.label.value_counts().to_dict() != {0:207, 1:207} or d.participant.nunique() != 11:
            raise ValueError(f"Window {w}: cohort does not match the manuscript.")
        if d.groupby("participant").label.nunique().min() != 2:
            raise ValueError("A participant lacks a class for within-person AUC.")

## Leave-one-participant-out evaluation
The training fold alone supplies imputation medians and scaling parameters. Random forests use 300 trees, maximum depth 8, minimum leaf size 3 and balanced class weights. Logistic regression is the comparator. The bootstrap below resamples participants, not individual windows. Its implementation and seed are reconstruction choices; compare the results rather than assuming numerical identity.

In [ ]:
def evaluate(d, cols, kind):
    X, y, g = d[cols], d.label.to_numpy(), d.participant.to_numpy()
    prob, pred = np.full(len(d), np.nan), np.full(len(d), -1)
    person_auc = []
    for tr, te in LeaveOneGroupOut().split(X, y, g):
        assert not set(g[tr]).intersection(g[te])
        if X.iloc[tr].isna().all().any():
            raise ValueError("Entire predictor missing in training fold.")
        if kind == "RF":
            model = make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(
                n_estimators=300, max_depth=8, min_samples_leaf=3,
                class_weight="balanced", random_state=2026, n_jobs=-1))
        else:
            model = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                LogisticRegression(max_iter=5000, class_weight="balanced", random_state=2026))
        model.fit(X.iloc[tr], y[tr])
        prob[te] = model.predict_proba(X.iloc[te])[:, 1]
        pred[te] = model.predict(X.iloc[te])
        person_auc.append(roc_auc_score(y[te], prob[te]))
    assert np.isfinite(prob).all() and (pred >= 0).all()
    means = np.random.default_rng(2026).choice(person_auc, (10000,len(person_auc))).mean(axis=1)
    lo, hi = np.quantile(means,[.025,.975])
    return dict(mean_auc=np.mean(person_auc), pooled_auc=roc_auc_score(y,prob),
                ci_low=lo, ci_high=hi, accuracy=accuracy_score(y,pred),
                f1=f1_score(y,pred), average_precision=average_precision_score(y,prob))

## Fresh results and discrepancy check
Writes aggregate results only, to a new run directory. No fresh outputs are created in reported-results mode.

In [ ]:
if path is not None:
    rows = [dict(window_minutes=w, model=m, **evaluate(data[data.window_minutes==w], features, m))
            for w in windows for m in ("RF","LR")]
    fits = pd.DataFrame(rows)
    out = Path("runs") / pd.Timestamp.now(tz="UTC").strftime("%Y%m%dT%H%M%S%fZ")
    out.mkdir(parents=True, exist_ok=False)
    fits.to_csv(out/"refit_summary.csv", index=False)
    display(fits)
    fresh = fits.query("model == 'RF'").set_index("window_minutes").mean_auc
    display(pd.concat([reported, fresh.rename("refit_mean_auc"), (fresh-reported).rename("difference")], axis=1))
    print("Fresh aggregate results:", out)
else:
    print("No feature table supplied. No model was fitted, and the original results have not been independently reproduced.")

## Optional: regenerate archived figures and tables
Original August export cell from ADARP_Continuous_Monitoring_Analysis.ipynb. It reads the archived result tables, not the new refit. It includes a separately labeled exploratory appendix. Outputs are redirected to runs/recovered_exports so the originals stay untouched.

In [ ]:
RUN_ARCHIVED_EXPORT = False
if RUN_ARCHIVED_EXPORT:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns

    from pathlib import Path

    # ============================================================
    # 1. LOCATE PROJECT
    # ============================================================

    possible_projects = [
        Path("/content/adarp_drive2/MyDrive/ADARP_Project"),
        Path("/content/adarp_drive3/MyDrive/ADARP_Project"),
        Path("/content/adarp_drive/MyDrive/ADARP_Project"),
        Path("/content/gdrive/MyDrive/ADARP_Project"),
        Path("/content/drive/MyDrive/ADARP_Project")
    ]

    PROJECT = next(
        (
            path for path in possible_projects
            if path.exists()
        ),
        None
    )

    if PROJECT is None:
        raise FileNotFoundError(
            "ADARP_Project was not found. Mount Google Drive first."
        )

    PROCESSED = PROJECT / "02_Processed_Data"
    RESULTS = PROJECT / "03_Results"
    FINAL = Path("runs") / "recovered_exports"
    FIGURES = FINAL / "Figures"
    TABLES = FINAL / "Tables"

    FIGURES.mkdir(parents=True, exist_ok=True)
    TABLES.mkdir(parents=True, exist_ok=True)

    print("Project:", PROJECT)

    # ============================================================
    # 2. LOAD FINAL RESULT FILES
    # ============================================================

    window_ci = pd.read_csv(
        RESULTS / "window_auc_confidence_intervals.csv"
    )

    window_comparisons = pd.read_csv(
        RESULTS / "window_pairwise_comparisons.csv"
    )

    modality_results = pd.read_csv(
        RESULTS / "modality_results_5min.csv"
    )

    importance = pd.read_csv(
        RESULTS / "permutation_importance_5min.csv"
    )

    participant_windows = pd.read_csv(
        RESULTS / "participant_auc_by_window.csv"
    )

    model_summary = pd.read_csv(
        RESULTS / "model_summary_core.csv"
    )

    participant_results = pd.read_csv(
        RESULTS / "participant_fold_results_core.csv"
    )

    next_day_results = pd.read_csv(
        RESULTS / "next_day_alcohol_model_summary.csv"
    )

    clean_data = pd.read_csv(
        PROCESSED / "clean_modeling_dataset.csv"
    )

    # Convert relevant columns to numeric.
    for frame in [
        window_ci,
        modality_results,
        importance,
        model_summary,
        participant_results,
        next_day_results
    ]:
        for column in frame.columns:
            if column not in [
                "participant",
                "model",
                "feature",
                "feature_set",
                "modality",
                "analysis"
            ]:
                frame[column] = pd.to_numeric(
                    frame[column],
                    errors="ignore"
                )

    # ============================================================
    # 3. PROFESSIONAL FIGURE STYLE
    # ============================================================

    sns.set_theme(
        style="whitegrid",
        context="paper"
    )

    plt.rcParams.update({
        "font.family": "DejaVu Sans",
        "font.size": 10,
        "axes.titlesize": 14,
        "axes.labelsize": 11,
        "axes.titleweight": "bold",
        "figure.dpi": 120,
        "savefig.dpi": 300,
        "axes.spines.top": False,
        "axes.spines.right": False
    })

    NAVY = "#183B56"
    BLUE = "#2C7FB8"
    TEAL = "#27A6A1"
    LIGHT_BLUE = "#A6CEE3"
    GOLD = "#E2A93B"
    GRAY = "#6B7280"
    RED = "#C84C4C"


    def save_figure(figure, filename):
        png_file = FIGURES / f"{filename}.png"
        pdf_file = FIGURES / f"{filename}.pdf"

        figure.savefig(
            png_file,
            bbox_inches="tight",
            facecolor="white"
        )

        figure.savefig(
            pdf_file,
            bbox_inches="tight",
            facecolor="white"
        )

        plt.close(figure)

        print("Saved:", png_file.name)
        print("Saved:", pdf_file.name)


    # ============================================================
    # 4. FIGURE 1 — PERFORMANCE BY WINDOW
    # ============================================================

    window_ci = window_ci.sort_values(
        "window_minutes"
    ).copy()

    x = window_ci["window_minutes"].to_numpy()
    y = window_ci["participant_mean_auc"].to_numpy()

    lower_error = (
        y - window_ci["ci_95_lower"].to_numpy()
    )

    upper_error = (
        window_ci["ci_95_upper"].to_numpy() - y
    )

    fig, ax = plt.subplots(figsize=(7.2, 4.6))

    ax.errorbar(
        x,
        y,
        yerr=[lower_error, upper_error],
        marker="o",
        markersize=8,
        linewidth=2.5,
        capsize=5,
        color=BLUE,
        markerfacecolor="white",
        markeredgewidth=2.5
    )

    ax.axhline(
        0.50,
        linestyle="--",
        linewidth=1.5,
        color=GRAY,
        label="Chance performance"
    )

    for x_value, y_value in zip(x, y):
        ax.annotate(
            f"{y_value:.3f}",
            (x_value, y_value),
            textcoords="offset points",
            xytext=(0, 10),
            ha="center",
            fontweight="bold"
        )

    ax.set_title(
        "Participant-Independent Performance by Monitoring Window"
    )

    ax.set_xlabel(
        "Physiological monitoring window before the reported event"
    )

    ax.set_ylabel(
        "Participant-mean ROC AUC"
    )

    ax.set_xticks(x)
    ax.set_xticklabels(
        [f"{int(value)} min" for value in x]
    )

    ax.set_ylim(0.40, 0.92)
    ax.legend(frameon=False, loc="upper right")

    fig.tight_layout()

    save_figure(
        fig,
        "Figure_1_AUC_by_monitoring_window"
    )

    # ============================================================
    # 5. FIGURE 2 — SIGNAL/MODALITY COMPARISON
    # ============================================================

    modality_plot = modality_results.copy()

    modality_name_map = {
        "All Core Signals": "All core signals",
        "All Core + HRV": "All core signals + HRV",
        "Heart Rate": "Heart rate",
        "EDA": "Electrodermal activity",
        "Temperature": "Skin temperature",
        "Movement": "Movement",
        "HRV Only": "Heart-rate variability"
    }

    modality_plot["display_name"] = (
        modality_plot["feature_set"]
        .replace(modality_name_map)
    )

    modality_plot = modality_plot.sort_values(
        "participant_mean_auc",
        ascending=True
    )

    fig, ax = plt.subplots(figsize=(7.5, 5.1))

    colors = [
        TEAL if "All core" in name
        else BLUE
        for name in modality_plot["display_name"]
    ]

    bars = ax.barh(
        modality_plot["display_name"],
        modality_plot["participant_mean_auc"],
        xerr=modality_plot["participant_sd_auc"],
        color=colors,
        alpha=0.90,
        capsize=4,
        edgecolor="white"
    )

    ax.axvline(
        0.50,
        linestyle="--",
        linewidth=1.5,
        color=GRAY
    )

    for bar, value in zip(
        bars,
        modality_plot["participant_mean_auc"]
    ):
        ax.text(
            min(value + 0.012, 0.965),
            bar.get_y() + bar.get_height() / 2,
            f"{value:.3f}",
            va="center",
            fontweight="bold"
        )

    ax.set_title(
        "Five-Minute Performance by Wearable Signal Set"
    )

    ax.set_xlabel(
        "Participant-mean ROC AUC (error bars show participant SD)"
    )

    ax.set_ylabel("")
    ax.set_xlim(0.40, 1.00)

    fig.tight_layout()

    save_figure(
        fig,
        "Figure_2_signal_modality_comparison"
    )

    # ============================================================
    # 6. FIGURE 3 — TOP PREDICTIVE FEATURES
    # ============================================================

    def readable_feature_name(feature):

        modality_map = {
            "hr": "Heart rate",
            "eda": "EDA",
            "temp": "Temperature",
            "acc": "Movement"
        }

        statistic_map = {
            "mean": "mean",
            "std": "variability",
            "median": "median",
            "min": "minimum",
            "max": "maximum",
            "slope": "trend"
        }

        parts = feature.replace(
            "_change",
            ""
        ).split("_")

        if len(parts) >= 2:
            modality = modality_map.get(
                parts[0],
                parts[0]
            )

            statistic = statistic_map.get(
                parts[1],
                parts[1]
            )

            return f"{modality}: {statistic} change"

        return feature


    top_features = (
        importance
        .sort_values(
            "mean_importance",
            ascending=False
        )
        .head(10)
        .sort_values(
            "mean_importance",
            ascending=True
        )
        .copy()
    )

    top_features["display_name"] = (
        top_features["feature"]
        .apply(readable_feature_name)
    )

    feature_color_map = {
        "Movement": BLUE,
        "Temperature": TEAL,
        "Heart Rate": GOLD,
        "EDA": RED
    }

    feature_colors = [
        feature_color_map.get(
            modality,
            GRAY
        )
        for modality in top_features["modality"]
    ]

    fig, ax = plt.subplots(figsize=(7.8, 5.5))

    ax.barh(
        top_features["display_name"],
        top_features["mean_importance"],
        xerr=top_features["sd_importance"],
        color=feature_colors,
        alpha=0.90,
        capsize=3,
        edgecolor="white"
    )

    ax.axvline(
        0,
        color=GRAY,
        linewidth=1
    )

    ax.set_title(
        "Most Predictive Features in the Five-Minute Model"
    )

    ax.set_xlabel(
        "Mean permutation importance across held-out participants"
    )

    ax.set_ylabel("")

    fig.tight_layout()

    save_figure(
        fig,
        "Figure_3_top_predictive_features"
    )

    # ============================================================
    # 7. FIGURE 4 — PARTICIPANT PERFORMANCE HEATMAP
    # ============================================================

    if {
        "participant",
        "window_minutes",
        "roc_auc"
    }.issubset(participant_windows.columns):

        heatmap_data = participant_windows.pivot(
            index="participant",
            columns="window_minutes",
            values="roc_auc"
        )

    else:
        participant_column = (
            "participant"
            if "participant" in participant_windows.columns
            else participant_windows.columns[0]
        )

        heatmap_data = participant_windows.set_index(
            participant_column
        )

        heatmap_data.columns = [
            pd.to_numeric(column, errors="ignore")
            for column in heatmap_data.columns
        ]

    heatmap_data = heatmap_data.reindex(
        sorted(
            heatmap_data.columns,
            key=lambda value: float(value)
        ),
        axis=1
    )

    heatmap_data.columns = [
        f"{int(float(column))} min"
        for column in heatmap_data.columns
    ]

    fig, ax = plt.subplots(figsize=(7.2, 5.7))

    sns.heatmap(
        heatmap_data,
        annot=True,
        fmt=".2f",
        cmap="Blues",
        vmin=0.20,
        vmax=1.00,
        linewidths=0.7,
        linecolor="white",
        cbar_kws={
            "label": "ROC AUC"
        },
        ax=ax
    )

    ax.set_title(
        "Performance for Each Held-Out Participant"
    )

    ax.set_xlabel("Monitoring window")
    ax.set_ylabel("Participant")

    fig.tight_layout()

    save_figure(
        fig,
        "Figure_4_participant_AUC_heatmap"
    )

    # ============================================================
    # 8. EXPLORATORY APPENDIX FIGURE — NEXT-DAY USE
    # ============================================================

    next_day_plot = next_day_results.copy()

    next_day_plot["display_name"] = (
        next_day_plot["analysis"]
        + "\n"
        + next_day_plot["model"]
        .replace({
            "Logistic Regression": "Logistic",
            "Random Forest": "Random Forest"
        })
    )

    next_day_plot = next_day_plot.sort_values(
        "roc_auc",
        ascending=False
    )

    fig, ax = plt.subplots(figsize=(8.5, 4.8))

    colors = [
        GOLD if analysis == "History only"
        else TEAL if analysis == "Wearable + history"
        else BLUE
        for analysis in next_day_plot["analysis"]
    ]

    bars = ax.bar(
        range(len(next_day_plot)),
        next_day_plot["roc_auc"],
        color=colors,
        alpha=0.90,
        edgecolor="white"
    )

    ax.axhline(
        0.50,
        linestyle="--",
        linewidth=1.5,
        color=GRAY,
        label="Chance performance"
    )

    for bar, value in zip(
        bars,
        next_day_plot["roc_auc"]
    ):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            value + 0.018,
            f"{value:.3f}",
            ha="center",
            fontweight="bold"
        )

    ax.set_xticks(range(len(next_day_plot)))
    ax.set_xticklabels(
        next_day_plot["display_name"],
        rotation=15,
        ha="right"
    )

    ax.set_ylim(0.20, 0.85)

    ax.set_ylabel("Leave-one-participant-out ROC AUC")

    ax.set_title(
        "Exploratory Next-Day Alcohol-Use Forecasting"
    )

    ax.legend(frameon=False)

    fig.tight_layout()

    save_figure(
        fig,
        "Appendix_Figure_1_next_day_alcohol_models"
    )

    # ============================================================
    # 9. EXPORT FINAL TABLES
    # ============================================================

    window_ci.to_csv(
        TABLES / "Table_1_window_performance.csv",
        index=False
    )

    window_comparisons.to_csv(
        TABLES / "Table_2_window_comparisons.csv",
        index=False
    )

    modality_results.to_csv(
        TABLES / "Table_3_modality_results.csv",
        index=False
    )

    importance.sort_values(
        "mean_importance",
        ascending=False
    ).to_csv(
        TABLES / "Table_4_feature_importance.csv",
        index=False
    )

    participant_results.to_csv(
        TABLES / "Table_5_participant_results.csv",
        index=False
    )

    next_day_results.to_csv(
        TABLES / "Appendix_Table_next_day_results.csv",
        index=False
    )

    # ============================================================
    # 10. FINAL PRIMARY-ANALYSIS SUMMARY
    # ============================================================

    primary_data = clean_data.loc[
        clean_data["window_minutes"] == 5
    ].copy()

    primary_model = model_summary.loc[
        (
            model_summary["window_minutes"] == 5
        )
        & (
            model_summary["model"]
            .str.contains(
                "Random Forest",
                case=False,
                na=False
            )
        )
    ].copy()

    primary_ci = window_ci.loc[
        window_ci["window_minutes"] == 5
    ].iloc[0]

    if len(primary_model) > 0:
        primary_model = primary_model.iloc[0]

        pooled_auc = primary_model.get(
            "roc_auc",
            np.nan
        )

        accuracy = primary_model.get(
            "accuracy",
            np.nan
        )

        f1 = primary_model.get(
            "f1",
            np.nan
        )

        average_precision = primary_model.get(
            "average_precision",
            np.nan
        )

    else:
        pooled_auc = np.nan
        accuracy = np.nan
        f1 = np.nan
        average_precision = np.nan

    key_statistics = pd.DataFrame([
        {
            "statistic": "Participants",
            "value": primary_data[
                "participant"
            ].nunique()
        },
        {
            "statistic": "Five-minute observations",
            "value": len(primary_data)
        },
        {
            "statistic": "Stress-related observations",
            "value": int(
                (
                    primary_data["label"] == 1
                ).sum()
            )
        },
        {
            "statistic": "Matched control observations",
            "value": int(
                (
                    primary_data["label"] == 0
                ).sum()
            )
        },
        {
            "statistic": "Core wearable features",
            "value": 24
        },
        {
            "statistic": "Pooled ROC AUC",
            "value": pooled_auc
        },
        {
            "statistic": "Participant-mean ROC AUC",
            "value": primary_ci[
                "participant_mean_auc"
            ]
        },
        {
            "statistic": "AUC 95% CI lower",
            "value": primary_ci[
                "ci_95_lower"
            ]
        },
        {
            "statistic": "AUC 95% CI upper",
            "value": primary_ci[
                "ci_95_upper"
            ]
        },
        {
            "statistic": "Accuracy",
            "value": accuracy
        },
        {
            "statistic": "F1 score",
            "value": f1
        },
        {
            "statistic": "Average precision",
            "value": average_precision
        }
    ])

    key_statistics.to_csv(
        TABLES / "Final_key_statistics.csv",
        index=False
    )

    print("\nFINAL PRIMARY RESULTS\n")
    print(
        key_statistics.to_string(
            index=False
        )
    )

    print("\nFINAL OUTPUT FOLDER")
    print(FINAL)

    print("\nFiles created:")

    for file in sorted(FINAL.rglob("*")):
        if file.is_file():
            print(file.relative_to(FINAL))